# Superstore Sales — Pareto (80/20) Analysis

**Question:** Given the `superstore_orders` table, identify which products contribute to the top 80% of total sales. Solve this in both SQL (using window functions) and PySpark (using DataFrame API with Window specs).

In [0]:
%sql
-- Explore the raw superstore_orders data
select * from b_sql.b_practice.superstore_orders

In [0]:
%sql
-- Calculate total sales and the 80% threshold value
-- total_sales   = sum of all sales across all orders
-- 80_p_sales    = 80% of total sales (the cumulative target for the Pareto analysis)
select sum(sales) as total_sales,sum(sales)*.8 from b_sql.b_practice.superstore_orders

In [0]:
%sql
-- SQL Solution: Find products contributing to the top 80% of total sales
--
-- CTE 1 (product_sales): Aggregate total sales per product, cast to decimal(10,4)
-- CTE 2 (sales_cal):    Compute running cumulative sales (ordered by p_sales desc)
--                        and the 80% threshold (sum of all p_sales * 0.8)
-- Final query:           Keep only products where running_sales < 80_p_sales
--                        (i.e., products that fit within the top 80% of total sales)
with product_sales as(select Product_ID , cast(sum(sales) as decimal(10,4)) as p_sales from b_sql.b_practice.superstore_orders group by Product_ID),
sales_cal as(
select *, sum(p_sales) over(order by p_sales desc rows between unbounded preceding and current row )as running_sales,(sum(p_sales) over())*.8 as 80_p_sales from product_sales)
select * from sales_cal where running_sales<80_p_sales


In [0]:
# Import common PySpark SQL functions, types, and Window specification
# (Window is needed for the cumulative running total in Cell 6)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Load the superstore_orders table from Unity Catalog into a Spark DataFrame
df_sales=spark.read.table("b_sql.b_practice.superstore_orders")
df_sales.display()

In [0]:
# PySpark Solution: Same Pareto 80% logic as the SQL CTE, using DataFrame API
#
# Step 1: Group by Product_ID and sum sales, order by total_sales descending
# Step 2: Define window w — ordered by total_sales desc, cumulative from first row to current
# Step 3: Define window w1 — no partition (whole dataset) for the grand total
# Step 4: Add running_total (cumulative sum over w) and 80_p_sales (grand total * 0.8)
# Step 5: Filter products where running_total < 80_p_sales (top 80% contributors)
df_sales_product=df_sales.groupBy("Product_ID").agg(sum("Sales").alias("total_sales")).orderBy(col("total_sales").desc())
w=Window.orderBy(col("total_sales").desc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)
w1=Window.partitionBy()

df_product=df_sales_product.withColumn("running_total",sum("total_sales").over(w))\
    .withColumn("80_p_sales",sum("total_sales").over(w1)*.8)
df_product.filter(col("running_total")<col("80_p_sales")).display()
